# Smart Portfolio Backtest

This notebook runs the core out-of-sample backtests for the portfolio strategies used in the project.

It compares:
- **Raw portfolios**, without volatility targeting.
- **Volatility-targeted portfolios**, scaled toward a 10% annual volatility target.

Transaction costs, risk-free cash returns, leverage constraints, and borrowing costs are included where relevant.


In [ ]:
# ============================================================
# IMPORTS
# ============================================================

import sys
sys.path.append("../src")

import importlib
import pandas as pd
import matplotlib.pyplot as plt

import metrics
import allocations
import backtesting

importlib.reload(metrics)
importlib.reload(allocations)
importlib.reload(backtesting)

In [ ]:
# ============================================================
# PARAMETERS
# ============================================================

WINDOW = 3 * 252
STEP = 21
TRANSACTION_COST = 0.001
VOL_TARGET = 0.10
MAX_LEVERAGE = 2.0
FUNDING_SPREAD = 0.01

## Data

Daily ETF prices are converted into returns. The annual risk-free rate is converted to a daily return and aligned with ETF trading dates.


In [ ]:
# ============================================================
# DATA
# ============================================================

prices = pd.read_csv("../data/etf_prices_raw.csv", index_col="Date", parse_dates=True)
returns = metrics.compute_returns(prices)

rf_rate = pd.read_csv("../data/risk_free_rate.csv", index_col="Date", parse_dates=True)["risk_free_rate"]
rf_rate = pd.to_numeric(rf_rate, errors="coerce").ffill()

annual_rf = rf_rate / 100
risk_free_returns = (1 + annual_rf) ** (1 / 252) - 1
risk_free_returns = risk_free_returns.reindex(returns.index).ffill()

print(f"Sample: {returns.index.min().date()} to {returns.index.max().date()}")
print(f"Trading days: {len(returns):,}")
print(f"Assets: {returns.shape[1]}")

## Strategies

The portfolio universe includes simple benchmarks, risk-based allocations, optimization-based allocations, and concentrated equity benchmarks.

| Strategy | Description |
|---|---|
| Equal Weight | Equal capital allocation across assets |
| Inverse Vol | Weights inversely proportional to asset volatility |
| Max Geo | Approximate geometric-growth optimization |
| Max Sharpe | Long-only Sharpe-ratio optimization |
| Carver | Handcrafted hierarchical risk allocation |
| Hierarchical Inv Vol | Inverse-volatility allocation within and across asset groups |
| SPY | U.S. equity benchmark |
| 60/40 | Traditional equity/bond benchmark |
| Tech | Technology-heavy equity benchmark |


In [ ]:
# ============================================================
# GROUPS AND STRATEGIES
# ============================================================

groups = {
    "equities": ["SPY", "QQQ", "EWU", "EZU", "EEM"],
    "bonds": ["IEF", "SHY"],
    "alternatives": ["GLD", "VNQ"]
}

strategies = {
    "Equal Weight": (backtesting.equal_strategy, None),
    "Inverse Vol": (backtesting.risk_strategy, None),
    "Max Geo": (backtesting.max_geo_strategy, None),
    "Max Sharpe": (backtesting.max_sharpe_strategy, None),
    "Carver": (backtesting.carver_handcrafting_strategy, groups),
    "Hierarchical Inv Vol": (backtesting.inv_handcrafting_strategy, groups),
    "SPY": (backtesting.spy_strategy, None),
    "60/40": (backtesting.sixty_forty_strategy, None),
    "Tech": (backtesting.tech_strategy, None)
}

In [ ]:
# ============================================================
# SUMMARY HELPERS
# ============================================================

def performance_summary(port_ret, risk_free_returns):
    return pd.Series({
        "Annual Return": metrics.annualize_returns(port_ret),
        "Annual Volatility": metrics.annualize_volatility(port_ret),
        "Sharpe Ratio": metrics.sharpe_ratio(port_ret, risk_free_returns),
        "Drawdown": metrics.drawdown(port_ret)
    })

def leverage_summary(leverage):
    return pd.Series({
        "Average Leverage": leverage.mean(),
        "Max Leverage": leverage.max(),
        "Min Leverage": leverage.min(),
        "Time at 2x": (leverage >= 2.0 - 1e-10).mean()
    })

def turnover_summary(turnover):
    return pd.Series({
        "Average Turnover": turnover.mean(),
        "Max Turnover": turnover.max()
    })

def transaction_cost_summary(costs):
    return pd.Series({
        "Total Cost": costs.sum(),
        "Average Cost per Rebalance": costs.mean(),
        "Max Cost per Rebalance": costs.max()
    })

## 1. Raw portfolios

The raw backtest evaluates each strategy using its natural allocation, without volatility targeting. Transaction costs are included.


In [ ]:
# ============================================================
# RAW BACKTEST
# ============================================================

raw_portfolio_returns = {}
raw_weight_histories = {}
raw_turnover_histories = {}
raw_transaction_cost_histories = {}

for name, (strategy, strategy_groups) in strategies.items():
    port_ret, weights, _, turnover, transaction_cost = backtesting.backtest(
        returns,
        window=WINDOW,
        step=STEP,
        allocation_func=strategy,
        groups=strategy_groups,
        vol_target=None,
        risk_free_returns=risk_free_returns,
        transaction_cost=TRANSACTION_COST,
        funding_spread=FUNDING_SPREAD
    )

    raw_portfolio_returns[name] = port_ret
    raw_weight_histories[name] = weights
    raw_turnover_histories[name] = turnover
    raw_transaction_cost_histories[name] = transaction_cost

In [ ]:
# ============================================================
# RAW RESULTS
# ============================================================

raw_summary = pd.DataFrame({
    name: performance_summary(port_ret, risk_free_returns)
    for name, port_ret in raw_portfolio_returns.items()
}).T

raw_turnover_summary = pd.DataFrame({
    name: turnover_summary(turnover)
    for name, turnover in raw_turnover_histories.items()
}).T

raw_transaction_cost_summary = pd.DataFrame({
    name: transaction_cost_summary(costs)
    for name, costs in raw_transaction_cost_histories.items()
}).T

print("RAW PORTFOLIOS")
display(raw_summary.style.format({
    "Annual Return": "{:.2%}",
    "Annual Volatility": "{:.2%}",
    "Sharpe Ratio": "{:.3f}",
    "Drawdown": "{:.2%}"
}))

print("\nRAW TURNOVER")
display(raw_turnover_summary.style.format({
    "Average Turnover": "{:.2%}",
    "Max Turnover": "{:.2%}"
}))

print("\nRAW TRANSACTION COSTS")
display(raw_transaction_cost_summary.style.format({
    "Total Cost": "{:.2%}",
    "Average Cost per Rebalance": "{:.4%}",
    "Max Cost per Rebalance": "{:.4%}"
}))

## 2. Volatility-targeted portfolios

Each strategy is scaled toward a **10% annual volatility target**, subject to a **2x leverage cap**.

Positive cash earns the risk-free rate. Borrowed capital pays the risk-free rate plus a **1% annual funding spread**.


In [ ]:
# ============================================================
# VOL-TARGETED BACKTEST
# ============================================================

targeted_portfolio_returns = {}
targeted_weight_histories = {}
targeted_leverage_histories = {}
targeted_turnover_histories = {}
targeted_transaction_cost_histories = {}

for name, (strategy, strategy_groups) in strategies.items():
    port_ret, weights, leverage, turnover, transaction_cost = backtesting.backtest(
        returns,
        window=WINDOW,
        step=STEP,
        allocation_func=strategy,
        groups=strategy_groups,
        vol_target=VOL_TARGET,
        max_leverage=MAX_LEVERAGE,
        risk_free_returns=risk_free_returns,
        transaction_cost=TRANSACTION_COST,
        funding_spread=FUNDING_SPREAD
    )

    targeted_portfolio_returns[name] = port_ret
    targeted_weight_histories[name] = weights
    targeted_leverage_histories[name] = leverage
    targeted_turnover_histories[name] = turnover
    targeted_transaction_cost_histories[name] = transaction_cost

In [ ]:
# ============================================================
# VOL-TARGETED RESULTS
# ============================================================

targeted_summary = pd.DataFrame({
    name: performance_summary(port_ret, risk_free_returns)
    for name, port_ret in targeted_portfolio_returns.items()
}).T

targeted_leverage_summary = pd.DataFrame({
    name: leverage_summary(leverage)
    for name, leverage in targeted_leverage_histories.items()
}).T

targeted_turnover_summary = pd.DataFrame({
    name: turnover_summary(turnover)
    for name, turnover in targeted_turnover_histories.items()
}).T

targeted_transaction_cost_summary = pd.DataFrame({
    name: transaction_cost_summary(costs)
    for name, costs in targeted_transaction_cost_histories.items()
}).T

print("VOL-TARGETED PORTFOLIOS")
display(targeted_summary.style.format({
    "Annual Return": "{:.2%}",
    "Annual Volatility": "{:.2%}",
    "Sharpe Ratio": "{:.3f}",
    "Drawdown": "{:.2%}"
}))

print("\nVOL-TARGETED LEVERAGE")
display(targeted_leverage_summary.style.format({
    "Average Leverage": "{:.2f}x",
    "Max Leverage": "{:.2f}x",
    "Min Leverage": "{:.2f}x",
    "Time at 2x": "{:.2%}"
}))

print("\nVOL-TARGETED TURNOVER")
display(targeted_turnover_summary.style.format({
    "Average Turnover": "{:.2%}",
    "Max Turnover": "{:.2%}"
}))

print("\nVOL-TARGETED TRANSACTION COSTS")
display(targeted_transaction_cost_summary.style.format({
    "Total Cost": "{:.2%}",
    "Average Cost per Rebalance": "{:.4%}",
    "Max Cost per Rebalance": "{:.4%}"
}))

## 3. Wealth curves

The charts show the growth of $1 invested in each strategy. Raw and volatility-targeted portfolios are shown separately.


In [ ]:
# ============================================================
# WEALTH CURVES
# ============================================================

raw_returns_ts = pd.DataFrame(raw_portfolio_returns)
targeted_returns_ts = pd.DataFrame(targeted_portfolio_returns)

raw_wealth = (1 + raw_returns_ts).cumprod()
targeted_wealth = (1 + targeted_returns_ts).cumprod()

plt.figure(figsize=(14, 7))
for strategy in raw_wealth.columns:
    plt.plot(raw_wealth.index, raw_wealth[strategy], label=strategy)

plt.title("Growth of $1 Invested — Raw Portfolios")
plt.xlabel("Date")
plt.ylabel("Portfolio Value ($)")
plt.legend(ncol=3)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 7))
for strategy in targeted_wealth.columns:
    plt.plot(targeted_wealth.index, targeted_wealth[strategy], label=strategy)

plt.title("Growth of $1 Invested — 10% Volatility Target")
plt.xlabel("Date")
plt.ylabel("Portfolio Value ($)")
plt.legend(ncol=3)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Drawdown comparison

Drawdown curves show peak-to-trough losses over time for a representative subset of volatility-targeted strategies.


In [ ]:
# ============================================================
# DRAWDOWN CURVES
# ============================================================

selected = ["Max Geo", "Max Sharpe", "Carver", "SPY", "Tech"]

wealth_selected = targeted_wealth[selected]
running_max = wealth_selected.cummax()
drawdowns = wealth_selected / running_max - 1

plt.figure(figsize=(14, 7))
for strategy in selected:
    plt.plot(drawdowns.index, drawdowns[strategy], label=strategy)

plt.title("Drawdowns — Volatility-Targeted Portfolios")
plt.xlabel("Date")
plt.ylabel("Drawdown")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()